In [ ]:
import os
from contextlib import closing

import polars as pl
import psycopg2
from dotenv import load_dotenv

load_dotenv()

dsn = (
  f"host=localhost dbname={os.environ["EXAMPLE_DB"]} "
  f"user={os.environ["POSTGRES_USER"]} "
  f"password={os.environ["POSTGRES_PASSWORD"]}"
)

In [ ]:
with closing(psycopg2.connect(dsn)) as conn:
    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
SELECT *
FROM customers;
"""
            )
            print(cur.fetchall())
            cur.execute(
"""
SELECT *
FROM orders;
"""
            )
            print(cur.fetchall())

In [ ]:
with closing(psycopg2.connect(dsn)) as conn:
    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
SELECT *
FROM customers c
INNER JOIN orders o
ON c.id = o.customer_id;
"""
            )
            print(cur.fetchall())

In [ ]:
sql = """
SELECT *
FROM customers c
INNER JOIN orders o
ON c.id = o.customer_id;
"""

with closing(psycopg2.connect(dsn)) as conn:
    pl.read_database(
        sql,
        conn,
    ).show()

In [ ]:
sql = """
SELECT *
FROM customers c
LEFT JOIN orders o
ON c.id = o.customer_id;
"""

with closing(psycopg2.connect(dsn)) as conn:
    pl.read_database(
        sql,
        conn,
    ).show()

In [ ]:
sql = """
SELECT *
FROM customers c
RIGHT JOIN orders o
ON c.id = o.customer_id;
"""

with closing(psycopg2.connect(dsn)) as conn:
    pl.read_database(
        sql,
        conn,
    ).show()

In [ ]:
sql = """
SELECT *
FROM customers c
FULL JOIN orders o
ON c.id = o.customer_id;
"""

with closing(psycopg2.connect(dsn)) as conn:
    df = pl.read_database(
        sql,
        conn,
    )
    print(df)

In [ ]:
sql = """
SELECT *
FROM customers c
LEFT JOIN orders o
ON c.id = o.customer_id
WHERE o.order_id IS NULL;
"""

with closing(psycopg2.connect(dsn)) as conn:
    df = pl.read_database(
        sql,
        conn,
    )
    print(df)

In [ ]:
sql = """
SELECT *
FROM customers c
RIGHT JOIN orders o
ON c.id = o.customer_id
WHERE c.id IS NULL;
"""

with closing(psycopg2.connect(dsn)) as conn:
    df = pl.read_database(
        sql,
        conn,
    )
    print(df)

In [ ]:
sql = """
SELECT *
FROM customers c
FULL JOIN orders o
ON c.id = o.customer_id
WHERE c.id       IS NULL
   OR o.order_id IS NULL;
"""

with closing(psycopg2.connect(dsn)) as conn:
    df = pl.read_database(
        sql,
        conn,
    )
    print(df)

In [ ]:
sql = """
SELECT *
FROM customers c
LEFT JOIN orders o
    ON c.id = o.customer_id
WHERE o.customer_id IS NOT NULL;
"""

with closing(psycopg2.connect(dsn)) as conn:
    df = pl.read_database(
        sql,
        conn,
    )
    print(df)

In [ ]:
load_dotenv()

dsn_salesdb = (
    f"host=127.0.0.1 port=5432 dbname={os.environ["SALES_DB"]} "
    f"user={os.environ["POSTGRES_USER"]} "
    f"password={os.environ["POSTGRES_PASSWORD"]}"
)

In [ ]:
with closing(psycopg2.connect(dsn_salesdb)) as conn:
    with conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1")
            print(cur.fetchall())

In [ ]:
sql = """
SELECT
    o.orderid
  , CONCAT_WS(' ', c.firstname, c.lastname) AS customer_name
  , p.product                               AS product_name
  , o.quantity * o.sales                    AS sales_amount
  , p.price                                 AS product_price
  , CONCAT_WS(' ', e.firstname, e.lastname) AS salesperson_name
FROM sales.orders o
LEFT JOIN sales.customers c
    ON o.customerid = c.customerid
LEFT JOIN sales.products p
    ON o.productid = p.productid
LEFT JOIN sales.employees e
    ON o.salespersonid = e.employeeid
ORDER BY o.orderid;
"""

with closing(psycopg2.connect(dsn_salesdb)) as conn:
    df = pl.read_database(sql, conn)
    print(df)